# Module 9 — Errors, Debugging, and Testing

**Part II: Core Skills** · Level: Intermediate · Prerequisites: Module 8

## What you'll learn

- Handling exceptions with `try` / `except` / `else` / `finally`
- Raising exceptions and defining your own
- The exception hierarchy
- Practical debugging techniques
- Writing your first automated tests

---

## 9.1 What an exception is

When Python hits a problem it cannot resolve, it **raises an exception** and unwinds the call stack. Unhandled, it ends the program with a *traceback* — read it from the bottom up: the last line names the error, the lines above show where it happened.

| Exception | Typical cause |
|---|---|
| `ValueError` | right type, wrong value: `int("abc")` |
| `TypeError` | wrong type: `"a" + 1` |
| `ZeroDivisionError` | `1 / 0` |
| `IndexError` | list index out of range |
| `KeyError` | missing dictionary key |
| `AttributeError` | object has no such attribute |
| `FileNotFoundError` | file does not exist |

## 9.2 `try` / `except`

In [ ]:
try:
    print(1 / 0)
except ZeroDivisionError:
    print("division by zero")

try:
    value = int("abc")
except ValueError:
    print("bad integer")

Catch **specific** exceptions; a bare `except:` also traps `KeyboardInterrupt` and hides bugs. Bind the exception object with `as`:

In [ ]:
try:
    int("abc")
except ValueError as err:
    print(str(err))              # the message explains what went wrong
    print(type(err).__name__)    # ValueError

Handle several types together or separately:

In [ ]:
def parse(text):
    try:
        return int(text)
    except (ValueError, TypeError) as err:
        print(f"cannot parse {text!r}: {err}")
        return None

print(parse("12"), parse("x"), parse(None))

## 9.3 `else` and `finally`

- `else` runs only when **no** exception occurred.
- `finally` **always** runs — ideal for cleanup.

In [ ]:
try:
    print("trying")
except Exception:
    print("error")
else:
    print("no error")
finally:
    print("cleanup")

A fuller example that handles several failure modes:

In [ ]:
def safe_transfer(accounts: list[float], source: int, target: int, amount: float) -> None:
    try:
        if source == target:
            raise ValueError(f"source ({source}) and target ({target}) must differ")
        if accounts[source] < amount:
            raise ValueError("insufficient funds")
        accounts[source] -= amount
        accounts[target] += amount
    except IndexError as err:
        print(f"[IndexError] no such account -> {err}")
    except ValueError as err:
        print(f"[ValueError] invalid transfer -> {err}")
    else:
        print(f"transferred {amount} from {source} to {target}")
    finally:
        print("transfer attempt complete")

balances = [100.0, 50.0]
safe_transfer(balances, 0, 1, 30)    # works
safe_transfer(balances, 0, 0, 10)    # ValueError
safe_transfer(balances, 0, 5, 10)    # IndexError
print(balances)

## 9.4 Raising exceptions

Use `raise` to signal that your function cannot do its job.

In [ ]:
try:
    raise ValueError("bad input")
except ValueError as e:
    print(e)

def set_age(age):
    if not isinstance(age, int):
        raise TypeError("age must be an integer")
    if age < 0:
        raise ValueError("age cannot be negative")
    return age

for candidate in (30, -1, "x"):
    try:
        print(set_age(candidate))
    except (TypeError, ValueError) as e:
        print(type(e).__name__, "->", e)

Re-raise inside a handler with bare `raise`, or chain a cause with `raise NewError(...) from err`.

In [ ]:
def load_port(text):
    try:
        return int(text)
    except ValueError as err:
        raise RuntimeError(f"invalid port setting {text!r}") from err

try:
    load_port("eighty")
except RuntimeError as e:
    print(e, "| caused by:", repr(e.__cause__))

## 9.5 Custom exceptions

Subclass `Exception` to give your own errors meaningful names. A shared base class lets callers catch a whole family at once.

In [ ]:
class AppError(Exception):
    """Base class for all errors in this application."""

class ValidationError(AppError):
    pass

class NotFoundError(AppError):
    def __init__(self, item):
        super().__init__(f"{item!r} was not found")
        self.item = item

try:
    raise NotFoundError("invoice-17")
except AppError as e:                 # catches any AppError subclass
    print(type(e).__name__, e, e.item)

## 9.6 The exception hierarchy

Built-in exceptions form a tree rooted at `BaseException`. `except Exception` catches nearly everything you would want to handle; `except LookupError` catches both `IndexError` and `KeyError`.

In [ ]:
print(IndexError.__mro__)
print(issubclass(KeyError, LookupError), issubclass(ZeroDivisionError, ArithmeticError))

try:
    raise TypeError("bad type")
except Exception as e:
    print(type(e).__name__)

Order `except` clauses from **most specific to most general**; the first match wins.

## 9.7 Debugging

Debugging means observing values and flow until you find where reality diverges from your expectation.

In [ ]:
x = 3
print("debug: x =", x)                 # quick and effective
print(f"{x=}")                         # shows the expression and its value

import traceback
try:
    {}["missing"]
except KeyError:
    print(traceback.format_exc().splitlines()[-1])

A reliable routine:

1. **Reproduce** the problem with the smallest input you can.
2. **Read** the traceback bottom-up.
3. **Form a hypothesis** and check it with a `print`, an `assert`, or the debugger.
4. **Fix one thing** and re-run.

For step-through debugging, call `breakpoint()` to drop into `pdb`, or use your editor's debugger.

## 9.8 Assertions

`assert condition, message` raises `AssertionError` if the condition is false. Use assertions to document *internal assumptions* and in tests — not to validate user input, because assertions can be disabled with `python -O`.

In [ ]:
x = 5
assert x > 0
print("ok")

def normalise(weights):
    total = sum(weights)
    assert total > 0, "weights must sum to a positive number"
    return [w / total for w in weights]

print(normalise([1, 1, 2]))

## 9.9 Testing basics

A **test** runs your code on known inputs and checks the output. Start with plain `assert`; graduate to `unittest` (built in) or `pytest`.

In [ ]:
def add(a, b):
    return a + b

assert add(2, 3) == 5
assert add(-1, 1) == 0
print("tests passed")

The same idea with `unittest`, including a test that expects an exception:

In [ ]:
import unittest

def divide(a, b):
    if b == 0:
        raise ValueError("division by zero")
    return a / b

class DivideTests(unittest.TestCase):
    def test_normal(self):
        self.assertEqual(divide(10, 2), 5)

    def test_float_result(self):
        self.assertAlmostEqual(divide(1, 3), 0.3333, places=3)

    def test_zero(self):
        with self.assertRaises(ValueError):
            divide(1, 0)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(DivideTests)
result = unittest.TextTestRunner(verbosity=0).run(suite)
print("all passed:", result.wasSuccessful())

In a real project you would save tests in `test_*.py` files and run `python -m unittest` (or `pytest`). Module 19 does exactly this.

## Common pitfalls

- A bare `except:` or `except Exception: pass` that silently swallows bugs.
- Catching an exception too early, where you cannot do anything useful about it.
- Putting too much code inside `try`; keep the `try` block small.
- Using `assert` for input validation.

## Practice

1. Write `read_int(prompt_text)` that keeps asking until the user enters a valid integer (simulate input with a list).
2. Create `InsufficientFundsError(AppError)` and raise it from a `withdraw()` function.
3. Write three `unittest` tests for a `clamp(value, low, high)` function, including one edge case.